# Notebook 01: Data Import, QC & Preprocessing

This notebook covers Steps 1-7 of the microbiome-metabolome pipeline.

**Dataset scope:** the three cohorts used downstream, `utils.DATASETS_ANALYSIS` =
`["YACHIDA-CRC-2019", "SINHA_CRC_2016", "Kim_adenomas_2020"]`. Yachida is the **discovery**
cohort (shotgun species, full metabolome, CRC staging, complete confounders); Sinha and Kim
are **external validation** cohorts (genus-level 16S, pre-scaled metabolomes, binary
Healthy/CRC). The IBD/IBS/ESRD/gastric cohorts are not loaded: nothing downstream used them,
and loading all 7 raw tables exceeded the server's 1.5 GB memory cap.

`ERAWIJANTARI-GASTRIC-CANCER-2020` is excluded from the study on principle, not just for
scope: 53 of its samples are the *same* Healthy subjects as Yachida's, so it is not an
independent cohort and using it would double-count controls.

## Group policy (v2.1)

| Yachida group | n | Treatment |
|---|---|---|
| Healthy | 127 | primary |
| Stage_0 + Stage_I_II | 96 | primary, `Early_CRC` |
| Stage_III_IV | 54 | primary, `Advanced_CRC` |
| **HS** (healthy after surgery) | 30 | kept as **`PostSurgery`** — sensitivity analyses only |
| **MP** (multiple polypoid adenomas) | 40 | **excluded from the study** |

- **MP is dropped at load time** by `utils.load_all_datasets()`
  (`utils.EXCLUDED_STUDY_GROUPS`), so no downstream notebook ever sees an adenoma sample.
  Kim's own `Adenoma` arm is dropped for the same reason, leaving Control vs Carcinoma.
- **HS is loaded but quarantined.** These patients had colorectal cancer resected and are
  currently cancer-free, so a resected gut is neither a clean control (altered anatomy and
  transit) nor an active tumour. Pooling it either way biases the comparison, so it is its
  own group and is excluded from every primary analysis by `utils.primary_mask()`.
  `PostSurgery` carries a **NaN** position on the ordinal severity scale, so it cannot enter
  a trend test even if a caller forgets to mask it.

Loaded: Yachida n=307. **Primary analysis set: n=277.**

**Pre-scaled metabolomes:** Sinha and Kim metabolites are already standardised upstream
(they contain negative values), so they skip prevalence filtering, half-min imputation,
log10 and median centring (`MTB_PRESCALED`, detected by `utils.is_prescaled()`).

**Polyamine proxies:** Kim does not measure putrescine or cadaverine directly, only their
N-acetyl derivatives. Those are recorded as explicit, **pre-specified proxies**
(`utils.POLYAMINE_PROXY_KEGG` / `_NAMES`) and kept in a separate `polyamine_proxies` dict —
never merged into `polyamine_columns` — because N-acetylation is an inactivation step, so a
derivative tracks its parent only loosely. Every downstream result built on a proxy is
labelled as such.

**Steps:**
1. Data Import -- load the three cohorts, validate sample/metabolite alignment, harmonize
   metadata, identify polyamine metabolites (+ proxies), build a metabolite-to-KEGG-ID map.
2. Quality Control -- missing values, sample-level QC (total signal, Shannon diversity),
   feature-level QC (detection rate), raw-data PCA.
3. Taxonomic Name Reduction -- collapse duplicate species-level taxonomy strings.
4. Feature Reduction -- prevalence + near-zero-variance filtering
   (`utils.PREVALENCE_SPECIES` = 20%, `utils.PREVALENCE_MTB` = 15%), with polyamines protected.
5. Transformation & Normalisation -- CLR for species, half-min imputation + log10 + median
   centring for (non-pre-scaled) metabolites.
6. Global Association Tests (Mantel, RDA) -- exploratory.
7. Differential abundance per fine-grained study group -- exploratory (consumed by NB01b).
   Species-metabolite and cross-cohort correlations are done in Notebook 02.

Output: `RESULTS_DIR/intermediate/preprocessed_data_v2.pkl`, consumed by
`01b_yachida_significant_features.ipynb` and `02_analysis.ipynb`.

In [ ]:
import sys
sys.path.insert(0, '.')

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

import utils
from utils import (
    DATA_DIR, RESULTS_DIR,
    DATASETS, DATASETS_ALL, DATASETS_ANALYSIS, DATASETS_IBD,
    DATASET_DISCOVERY, DATASETS_VALIDATION,
    POLYAMINE_DATASETS_ANALYSIS, POLYAMINE_KEGG, COMPARISONS,
    CRC_STAGE_ORDER, CRC_STAGE_ORDER_WITH_SENSITIVITY,
    STAGE_GROUP_ORDER, STAGE_GROUP_ORDER_WITH_SENSITIVITY,
    PREVALENCE_SPECIES, PREVALENCE_MTB,
    load_all_datasets, harmonize_metadata,
    find_polyamine_columns, find_polyamine_columns_with_proxies,
    reduce_species_names,
    compute_sample_qc, compute_feature_qc,
    filter_by_prevalence, filter_near_zero_variance,
    clr_transform, log_transform, impute_half_min,
    validate_sample_alignment, validate_metabolite_alignment,
    print_dataset_summary, plot_pca,
    _draw_confidence_ellipse,
    differential_abundance, volcano_plot,
    primary_mask, sensitivity_mask,
)

# The three cohorts used downstream (see markdown above).
DATASETS = list(DATASETS_ANALYSIS)

# Create output directories (figures under figures/<topic>/ only)
for sub in ['figures/qc', 'figures/differential', 'figures/correlation',
            'figures/network', 'figures/ml']:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'tables').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'intermediate').mkdir(parents=True, exist_ok=True)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')
print(f'Discovery cohort : {DATASET_DISCOVERY}')
print(f'Validation cohorts: {DATASETS_VALIDATION}')
print(f'Excluded study groups (dropped at load): {utils.EXCLUDED_STUDY_GROUPS}')
print(f'Sensitivity groups (loaded, non-primary): {utils.SENSITIVITY_GROUPS}')
print(f'Prevalence thresholds -> species: {PREVALENCE_SPECIES:.0%}, metabolites: {PREVALENCE_MTB:.0%}')

---
## Step 1: Data Import

In [ ]:
# Load the analysis cohorts. Excluded groups (Yachida MP, Kim Adenoma) are dropped
# inside load_all_datasets, before any QC step, so nothing downstream can see them.
data = load_all_datasets(datasets=DATASETS)
print('Datasets loaded.')
summary = print_dataset_summary(data)
display(summary)

# Sinha and Kim metabolites are pre-standardised (negative values present): no
# impute/log/median-centring for them. Detected, not hardcoded per cohort.
MTB_PRESCALED = {ds: utils.is_prescaled(data[ds]['mtb']) for ds in DATASETS}
print('Metabolite table already scaled upstream:', MTB_PRESCALED)

### 1.1 Validate Sample ID Alignment

In [ ]:
alignment_reports = []
for ds in DATASETS:
    report = validate_sample_alignment(data, ds)
    alignment_reports.append({
        'Dataset':         ds,
        'Meta samples':    report['meta_n'],
        'Mtb samples':     report['mtb_n'],
        'Species samples': report['species_n'],
        'Common (all 3)':  len(report['common']),
        'Meta only':       len(report['meta_only']),
        'Mtb only':        len(report['mtb_only']),
        'Species only':    len(report['spc_only']),
    })
    if report['meta_only'] or report['mtb_only'] or report['spc_only']:
        print(f'WARNING: {ds} has misaligned samples')

display(pd.DataFrame(alignment_reports))

### 1.2 Validate Metabolite Alignment (mtb <-> mtb.map)

In [ ]:
for ds in DATASETS:
    r = validate_metabolite_alignment(data, ds)
    status = 'OK' if len(r['in_mtb_not_map']) == 0 and len(r['in_map_not_mtb']) == 0 else 'MISMATCH'
    print(f"{ds}: mtb={r['mtb_features']}, map={r['map_compounds']}, "
          f"in_mtb_not_map={len(r['in_mtb_not_map'])}, "
          f"in_map_not_mtb={len(r['in_map_not_mtb'])} [{status}]")

### 1.3 Harmonise Metadata

In [ ]:
harmonized_meta = {}
for ds in DATASETS:
    harmonized_meta[ds] = harmonize_metadata(data[ds]['metadata'], ds)

for ds in DATASETS:
    print(f'\n{ds}:')
    print(harmonized_meta[ds]['Study.Group'].value_counts().to_string())

# --- Yachida group accounting (the v2.1 policy, asserted not just described) ---
yachida_meta = harmonized_meta['YACHIDA-CRC-2019']
assert yachida_meta['Dataset'].notna().all()

# MP is excluded from the study entirely.
assert not yachida_meta['Study.Group'].eq('MP').any(), \
    'MP samples must be excluded at load time'
# HS is NOT excluded -- it is kept, relabelled PostSurgery, and quarantined.
n_hs = int(yachida_meta['Study.Group'].eq('HS').sum())
assert n_hs == 30, f'expected 30 HS (PostSurgery) samples retained, got {n_hs}'

print('\n--- Yachida fine-grained distribution (utils.CRC_STAGE_ORDER_WITH_SENSITIVITY) ---')
stage_dist = (yachida_meta['Study.Group'].value_counts()
              .reindex(['Healthy', 'HS', 'Stage_0', 'Stage_I_II', 'Stage_III_IV'], fill_value=0))
print(stage_dist.to_string())

# Grouped scheme used by NB02+ (utils.STAGE_GROUP_MAP): Healthy / Early_CRC /
# Advanced_CRC on the primary axis, plus PostSurgery off-axis.
print('\n--- Yachida Stage.Group distribution (grouped scheme) ---')
stage_group_dist = (yachida_meta['Stage.Group'].value_counts()
                    .reindex(STAGE_GROUP_ORDER_WITH_SENSITIVITY, fill_value=0))
print(stage_group_dist.to_string())

n_primary = int(primary_mask(yachida_meta).sum())
n_sens    = int(sensitivity_mask(yachida_meta).sum())
print(f'\nPrimary analysis set : n={n_primary}   (Healthy + Early_CRC + Advanced_CRC)')
print(f'Sensitivity set      : n={n_sens}   (PostSurgery / HS -- excluded from all primary analyses)')
print(f'Total loaded         : n={len(yachida_meta)}')
assert n_primary + n_sens == len(yachida_meta), 'primary + sensitivity must partition the cohort'

# PostSurgery must have no position on the ordinal severity scale, so that a
# forgotten mask cannot silently put it into a trend test.
assert yachida_meta.loc[yachida_meta['Stage.Group'] == 'PostSurgery',
                        'Stage.Group.Numeric'].isna().all(), \
    'PostSurgery must be NaN on the ordinal scale'
assert yachida_meta.loc[yachida_meta['Study.Group'] == 'HS',
                        'CRC.Stage.Numeric'].isna().all()
print('\nOrdinal-scale guard OK: PostSurgery is NaN in Stage.Group.Numeric and CRC.Stage.Numeric.')

In [ ]:
# Age, Gender, BMI distributions
for ds in DATASETS:
    hm = harmonized_meta[ds]
    print(f'\n{ds}:')
    print(f"  Age: mean={hm['Age'].mean():.1f}, "
          f"median={hm['Age'].median():.1f}, "
          f"range=[{hm['Age'].min():.0f}-{hm['Age'].max():.0f}], "
          f"missing={hm['Age'].isna().sum()}")
    if hm['Gender'].notna().any():
        print(f"  Gender: {hm['Gender'].value_counts().to_dict()}")
    if hm['BMI'].notna().any():
        print(f"  BMI: mean={hm['BMI'].mean():.1f}, missing={hm['BMI'].isna().sum()}")

### 1.4 Identify Polyamine Metabolites

In [ ]:
# Polyamine columns per cohort, plus pre-specified acetyl PROXIES for parents a
# cohort does not measure directly (kept separate -- see the markdown at the top).
polyamine_columns = {}
polyamine_proxies = {}

for ds in DATASETS:
    direct, proxies = find_polyamine_columns_with_proxies(
        data[ds]['mtb.map'], data[ds]['mtb'], ds)
    polyamine_columns[ds] = direct
    polyamine_proxies[ds] = proxies
    print(f'\n{ds}: {len(direct)} polyamines measured directly, {len(proxies)} via proxy')
    for kegg_id, col_name in direct.items():
        print(f'  [direct] {kegg_id} ({POLYAMINE_KEGG[kegg_id]}): {col_name}')
    for kegg_id, (col_name, via) in proxies.items():
        print(f'  [PROXY ] {kegg_id} ({POLYAMINE_KEGG[kegg_id]}): {col_name}  <- {via}')

# Coverage matrix: which cohort can speak to which polyamine, and how.
pa_summary = []
for kegg_id, name in POLYAMINE_KEGG.items():
    row = {'KEGG': kegg_id, 'Name': name}
    for ds in DATASETS:
        short = ds.split('-')[0].split('_')[0]
        if kegg_id in polyamine_columns[ds]:
            row[short] = 'direct'
        elif kegg_id in polyamine_proxies[ds]:
            row[short] = 'proxy'
        else:
            row[short] = '-'
    pa_summary.append(row)
pa_coverage = pd.DataFrame(pa_summary)
display(pa_coverage)
pa_coverage.to_csv(RESULTS_DIR / 'tables' / 'polyamine_cohort_coverage_v2.csv', index=False)
print('\nSaved: polyamine_cohort_coverage_v2.csv')
print('A "proxy" cell means the parent compound was NOT measured in that cohort; only an '
      'N-acetyl derivative was. Replication claims resting on a proxy are labelled '
      'throughout NB02/NB08/NB09.')

### 1.5 Build Metabolite -> KEGG ID Map

The cross-dataset "consistent correlation" step in Step 7 needs to recognise the *same*
metabolite across cohorts that use different metabolomics platforms / naming conventions
(e.g. Putrescine may appear as `"C00134_Putrescine(1,4-Butanediamine)"` in one dataset,
`"PUTRESCINE"` in another, and `"HILp_QI163__putrescine"` in a third). Matching those exact
column-name strings finds almost nothing real.

We build `mtb_kegg_map: {dataset: {mtb_column_name: kegg_id}}` from each dataset's own
`mtb.map` file (`Compound` -> `KEGG`) and pass it to
`utils.find_consistent_correlations(kegg_map=mtb_kegg_map, ...)` below, so matching happens
on the (platform-independent) KEGG compound ID instead.

In [ ]:
mtb_kegg_map = {}
for ds in DATASETS:
    mtb_map_df = data[ds]['mtb.map']
    if mtb_map_df is not None and 'Compound' in mtb_map_df.columns and 'KEGG' in mtb_map_df.columns:
        sub = mtb_map_df.dropna(subset=['KEGG'])
        mtb_kegg_map[ds] = dict(zip(sub['Compound'], sub['KEGG']))
    else:
        mtb_kegg_map[ds] = {}
    print(f'{ds}: {len(mtb_kegg_map[ds])} metabolite -> KEGG mappings')

---
## Step 2: Quality Control

### 2.1 Missing Values & Duplicates

In [ ]:
qc_summary = []
for ds in DATASETS:
    mtb = data[ds]['mtb']
    spc = data[ds]['species']
    mtb_features = [c for c in mtb.columns if c != 'Sample']
    spc_features = [c for c in spc.columns if c != 'Sample']
    qc_summary.append({
        'Dataset':             ds,
        'Mtb NaN count':       mtb[mtb_features].isna().sum().sum(),
        'Mtb zero %':          f"{(mtb[mtb_features] == 0).values.mean():.1%}",
        'Species NaN count':   spc[spc_features].isna().sum().sum(),
        'Species zero %':      f"{(spc[spc_features] == 0).values.mean():.1%}",
        'Mtb dup samples':     mtb['Sample'].duplicated().sum(),
        'Species dup samples': spc['Sample'].duplicated().sum(),
    })
display(pd.DataFrame(qc_summary))

### 2.2 Sample-Level QC

In [ ]:
sample_qc_mtb = {}
sample_qc_spc = {}

for ds in DATASETS:
    sample_qc_mtb[ds] = compute_sample_qc(data[ds]['mtb'], data_type='metabolite')
    sample_qc_spc[ds] = compute_sample_qc(data[ds]['species'], data_type='species')

print('Species row sum check (should be ~1.0):')
for ds in DATASETS:
    sums = sample_qc_spc[ds]['total_signal']
    print(f'  {ds}: mean={sums.mean():.4f}, min={sums.min():.4f}, max={sums.max():.4f}')

In [ ]:
# Metabolite total signal by study group -- highlight IBD/IBS/ESRD (exploratory-only) cohorts
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 5), squeeze=False)
axes = axes.flatten()
for i, ds in enumerate(DATASETS):
    qc = sample_qc_mtb[ds].merge(harmonized_meta[ds][['Sample', 'Study.Group']], on='Sample')
    sns.boxplot(data=qc, x='Study.Group', y='total_signal', ax=axes[i])
    axes[i].set_title(f'{ds}\nMetabolite Total Signal', fontsize=9)
    axes[i].tick_params(axis='x', rotation=45)
    if ds in DATASETS_IBD:
        axes[i].set_facecolor('#fff8f0')
        axes[i].set_title(f'{ds}\nMetabolite Total Signal [EXPLORATORY]', fontsize=8)
for j in range(len(DATASETS), len(axes)):
    axes[j].axis('off')
plt.suptitle('Metabolite Total Signal by Study Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'qc' / 'mtb_total_signal_by_group_v2.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shannon diversity
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 5), squeeze=False)
axes = axes.flatten()
for i, ds in enumerate(DATASETS):
    qc = sample_qc_spc[ds].merge(harmonized_meta[ds][['Sample', 'Study.Group']], on='Sample')
    sns.boxplot(data=qc, x='Study.Group', y='shannon_diversity', ax=axes[i])
    axes[i].set_title(f'{ds}\nShannon Diversity', fontsize=9)
    axes[i].tick_params(axis='x', rotation=45)
for j in range(len(DATASETS), len(axes)):
    axes[j].axis('off')
plt.suptitle('Species Shannon Diversity by Study Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'qc' / 'species_shannon_by_group_v2.png',
            dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Feature-Level QC

In [ ]:
feature_qc_mtb = {}
feature_qc_spc = {}
for ds in DATASETS:
    feature_qc_mtb[ds] = compute_feature_qc(data[ds]['mtb'])
    feature_qc_spc[ds] = compute_feature_qc(data[ds]['species'])

# These are the SAME thresholds used for the actual filtering step below
# (utils.PREVALENCE_SPECIES / utils.PREVALENCE_MTB) -- kept in sync here so this
# preview can't silently drift out of agreement with what Step 4 actually does.
print(f'Features passing prevalence filter '
      f'(species >= {PREVALENCE_SPECIES:.0%}, metabolites >= {PREVALENCE_MTB:.0%}):')
for ds in DATASETS:
    mtb_pass = (feature_qc_mtb[ds]['detection_rate'] >= PREVALENCE_MTB).sum()
    spc_pass = (feature_qc_spc[ds]['detection_rate'] >= PREVALENCE_SPECIES).sum()
    print(f'  {ds}: mtb {mtb_pass}/{len(feature_qc_mtb[ds])}, '
          f'species {spc_pass}/{len(feature_qc_spc[ds])}')

In [ ]:
# Species detection rate histograms
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 5), squeeze=False)
axes = axes.flatten()
for i, ds in enumerate(DATASETS):
    axes[i].hist(feature_qc_spc[ds]['detection_rate'], bins=50, alpha=0.7,
                 edgecolor='black', color='steelblue')
    axes[i].axvline(PREVALENCE_SPECIES, color='red', ls='--',
                     label=f'{PREVALENCE_SPECIES:.0%} threshold')
    axes[i].set_title(f'{ds}\nSpecies Detection Rate', fontsize=9)
    axes[i].legend(fontsize=7)
for j in range(len(DATASETS), len(axes)):
    axes[j].axis('off')
plt.suptitle('Species Feature Detection Rate Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'qc' / 'species_detection_rates_v2.png',
            dpi=150, bbox_inches='tight')
plt.show()

### 2.4 PCA on Raw Data (Outlier / Batch Check)

In [ ]:
for ds in DATASETS:
    fig, ax = plot_pca(
        data[ds]['mtb'], harmonized_meta[ds],
        title=f'{ds} -- Raw Metabolites PCA',
        save_path=RESULTS_DIR / 'figures' / 'qc' / f'{ds}_raw_mtb_pca_v2.png'
    )
    plt.show()
    plt.close()

---
## Step 3: Taxonomic Name Reduction

In [ ]:
reduced_species = {}
species_name_maps = {}
import gc

for ds in DATASETS:
    try:
        reduced_df, name_map = reduce_species_names(data[ds]['species'])
        reduced_species[ds] = reduced_df
        species_name_maps[ds] = name_map
        n_orig = len(data[ds]['species'].columns) - 1
        n_reduced = len(reduced_df.columns) - 1
        n_collapsed = sum(1 for v in name_map.values() if len(v) > 1)
        print(f'{ds}: {n_orig} -> {n_reduced} species '
              f'({n_collapsed} names had duplicates collapsed)')
        # Free the raw (very wide) species table -- the reduced copy replaces it.
        data[ds]['species'] = None
    except Exception as e:
        print(f'ERROR processing {ds}: {e}')
        raise
gc.collect()


---
## Step 4: Feature Reduction

In [ ]:
filtered_species = {}
filtered_mtb = {}

# Protected metabolites = the polyamines themselves AND their pre-specified proxies;
# a proxy that gets filtered out would silently remove a cohort's only handle on
# that compound.
protected_by_ds = {
    ds: list(polyamine_columns.get(ds, {}).values())
        + [col for col, _via in polyamine_proxies.get(ds, {}).values()]
    for ds in DATASETS
}

print(f"{'Dataset':<45} {'Species (before->after)':<30} {'Metabolites (before->after)'}")
print('-' * 105)

for ds in DATASETS:
    protected_mtb = protected_by_ds[ds]

    # Species: utils.PREVALENCE_SPECIES (20%) prevalence threshold
    spc_prev = filter_by_prevalence(reduced_species[ds], threshold=PREVALENCE_SPECIES)
    spc_filt = filter_near_zero_variance(spc_prev)
    filtered_species[ds] = spc_filt

    # Metabolites: utils.PREVALENCE_MTB (15%) prevalence threshold; protect polyamines.
    # Pre-scaled (standardised) tables have no "detected = >0" meaning: skip prevalence.
    if MTB_PRESCALED[ds]:
        mtb_prev = data[ds]['mtb']
    else:
        mtb_prev = filter_by_prevalence(data[ds]['mtb'], threshold=PREVALENCE_MTB,
                                        protected_cols=protected_mtb)
    mtb_filt = filter_near_zero_variance(mtb_prev)
    filtered_mtb[ds] = mtb_filt

    n_spc_before = len(reduced_species[ds].columns) - 1
    n_spc_after = len(spc_filt.columns) - 1
    n_mtb_before = len(data[ds]['mtb'].columns) - 1
    n_mtb_after = len(mtb_filt.columns) - 1
    print(f"{ds:<45} {n_spc_before:>6} -> {n_spc_after:<6} "
          f"({n_spc_after / n_spc_before:.1%})      "
          f"{n_mtb_before:>6} -> {n_mtb_after:<6} ({n_mtb_after / n_mtb_before:.1%})")

In [ ]:
# Sparsity check after filtering
print('Sparsity after filtering (% zeros in filtered species):')
for ds in DATASETS:
    spc_cols = [c for c in filtered_species[ds].columns if c != 'Sample']
    sparsity = (filtered_species[ds][spc_cols] == 0).values.mean()
    n_features = len(spc_cols)
    flag = ' -> High sparsity! Consider genus-level aggregation.' if sparsity > 0.8 else ''
    print(f'  {ds}: {sparsity:.1%} zeros, {n_features} features{flag}')

In [ ]:
# Confirm every polyamine (and proxy) survived filtering. A LOST column is a hard
# error, not a warning: it would silently drop a compound from the whole study.
print('Polyamine preservation check after filtering:')
lost = []
for ds in POLYAMINE_DATASETS_ANALYSIS:
    mtb_cols = set(filtered_mtb[ds].columns)
    for kegg_id, col_name in polyamine_columns[ds].items():
        ok = col_name in mtb_cols
        print(f'  {ds:22s} | {POLYAMINE_KEGG[kegg_id]:11s} [direct] : {"KEPT" if ok else "LOST"}')
        if not ok:
            lost.append((ds, POLYAMINE_KEGG[kegg_id], 'direct', col_name))
    for kegg_id, (col_name, via) in polyamine_proxies[ds].items():
        ok = col_name in mtb_cols
        print(f'  {ds:22s} | {POLYAMINE_KEGG[kegg_id]:11s} [proxy ] : {"KEPT" if ok else "LOST"}')
        if not ok:
            lost.append((ds, POLYAMINE_KEGG[kegg_id], 'proxy', col_name))

assert not lost, f'protected polyamine column(s) lost during filtering: {lost}'
print('\nAll polyamine and proxy columns retained.')

---
## Step 5: Transformation & Normalisation

In [ ]:
# CLR transform species (pseudocount=1e-4 to avoid extreme negative values)
transformed_species = {}
for ds in DATASETS:
    transformed_species[ds] = clr_transform(filtered_species[ds], pseudocount=1e-4)

# Log10 transform metabolites: half-minimum NaN imputation (utils.impute_half_min,
# which correctly takes the minimum over NONZERO values only) -> log10(x + 1)
# (utils.log_transform(base=10)) -> sample-wise median centering.
transformed_mtb = {}
for ds in DATASETS:
    mtb_df = filtered_mtb[ds]
    pa_cols = protected_by_ds[ds]

    if MTB_PRESCALED[ds]:
        # Already standardised upstream: log10(x+1) would turn every value < -1 into
        # NaN, which is exactly the bug that silently voided Sinha in the old pipeline.
        transformed_mtb[ds] = mtb_df.copy()
        n_nan_pre = int(mtb_df.drop(columns='Sample').isna().sum().sum())
        print(f'  {ds}: pre-scaled metabolites used as-is (no impute/log/centring); NaN cells: {n_nan_pre}')
        continue

    n_nan = mtb_df.drop(columns='Sample').isna().sum().sum()
    imputed_df = impute_half_min(mtb_df, protected_cols=pa_cols)
    if n_nan > 0:
        print(f'  {ds}: imputed {n_nan} NaN values with half-minimum (utils.impute_half_min)')

    log_df = log_transform(imputed_df, pseudocount=1.0, base=10)

    # Sample-wise median centering
    t_cols = [c for c in log_df.columns if c != 'Sample']
    X = log_df[t_cols].values
    X = X - np.nanmedian(X, axis=1, keepdims=True)
    log_df[t_cols] = X

    transformed_mtb[ds] = log_df

# Post-transform guard: a pre-scaled table must not have acquired NaNs, and a
# log-transformed one must be finite everywhere.
for ds in DATASETS:
    vals = transformed_mtb[ds].drop(columns='Sample').to_numpy(dtype=float)
    n_bad = int((~np.isfinite(vals)).sum())
    assert n_bad == 0, f'{ds}: {n_bad} non-finite metabolite cells after transformation'

print('Transformations applied:')
print('  Species     : CLR (pseudocount=1e-4)')
print('  Metabolites : half-min NaN imputation -> log10(x+1) -> sample-wise median centering')
print(f'                (skipped for pre-scaled tables: {[d for d, v in MTB_PRESCALED.items() if v]})')
print('  All metabolite matrices finite: OK')

In [ ]:
# Sanity check: CLR value range should not have extreme negatives
print('CLR value range check (values below -10 flag pseudocount issues):')
for ds in DATASETS:
    spc_cols = [c for c in transformed_species[ds].columns if c != 'Sample']
    vals = transformed_species[ds][spc_cols].values
    pct_extreme = (vals < -10).mean()
    print(f'  {ds}: min={vals.min():.2f}, max={vals.max():.2f}, '
          f'% below -10: {pct_extreme:.2%}')

In [ ]:
# PCA on transformed species
for ds in DATASETS:
    fig, ax = plot_pca(
        transformed_species[ds], harmonized_meta[ds],
        title=f'{ds} -- CLR-Transformed Species PCA',
        save_path=RESULTS_DIR / 'figures' / 'qc' / f'{ds}_clr_species_pca_v2.png'
    )
    plt.show()
    plt.close()

In [ ]:
# PCA on transformed metabolites (log10, NOT log2 -- utils.log_transform(base=10) above)
for ds in DATASETS:
    fig, ax = plot_pca(
        transformed_mtb[ds], harmonized_meta[ds],
        title=f'{ds} -- Log10-Transformed Metabolites PCA',
        save_path=RESULTS_DIR / 'figures' / 'qc' / f'{ds}_log10_mtb_pca_v2.png'
    )
    plt.show()
    plt.close()

---
## Step 6: Global Association Tests (Mantel, RDA)

> **Exploratory section -- the two analysis cohorts (Yachida, Sinha).**
> Confirmatory analysis is in Notebook 02.

In [ ]:
# -- Exploratory flag --------------------------------------------------------
# Set EXPLORATORY = False to skip the cross-cohort exploratory analyses below
# (Mantel/RDA in Step 6, differential abundance in Step 7). They run on the two
# analysis cohorts; NB01b needs the Step 7 DA results, so keep this True for the
# pipeline. The final save cell is guarded either way.
EXPLORATORY = True

from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler


def mantel_test(X, Y, n_perms=999, metric='euclidean'):
    '''Mantel test: correlation between two distance matrices.'''
    dist_X = pdist(X, metric=metric)
    dist_Y = pdist(Y, metric=metric)
    r_obs, _ = pearsonr(dist_X, dist_Y)
    n = X.shape[0]
    sq_Y = squareform(dist_Y)
    r_null = np.zeros(n_perms)
    rng = np.random.default_rng(42)
    for i in range(n_perms):
        perm = rng.permutation(n)
        sq_perm = sq_Y[np.ix_(perm, perm)]
        r_null[i], _ = pearsonr(dist_X, squareform(sq_perm))
    p_value = (np.sum(r_null >= r_obs) + 1) / (n_perms + 1)
    return r_obs, p_value, r_null


print('Mantel test function defined.')

In [ ]:
if EXPLORATORY:
    print('=' * 80)
    print(f'MANTEL TEST: Global microbiome-metabolome association (all {len(DATASETS)} datasets)')
    print('=' * 80)

    mantel_results = []
    for ds in DATASETS:
        spc = transformed_species[ds].set_index('Sample')
        mtb = transformed_mtb[ds].set_index('Sample')
        common = spc.index.intersection(mtb.index)
        X_spc = np.nan_to_num(spc.loc[common].values, nan=0.0)
        X_mtb = np.nan_to_num(mtb.loc[common].values, nan=0.0)

        r_obs, p_val, _ = mantel_test(X_spc, X_mtb, n_perms=999)
        sig_label = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        is_ibd = '(IBD/IBS -- exploratory)' if ds in DATASETS_IBD else ''

        mantel_results.append({
            'Dataset': ds, 'Mantel r': round(r_obs, 4),
            'p-value': round(p_val, 4), 'Significant': 'Yes' if p_val < 0.05 else 'No',
            'n_samples': len(common), 'Note': is_ibd,
        })
        print(f'  {ds}: r={r_obs:.4f}, p={p_val:.4f} {sig_label} {is_ibd}')

    display(pd.DataFrame(mantel_results))

In [ ]:
from sklearn.decomposition import PCA as _PCA


def rda_analysis(X_species, Y_metab, n_components=2):
    '''Redundancy Analysis: constrained ordination of Y explained by X.'''
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    X = scaler_x.fit_transform(X_species)
    Y = scaler_y.fit_transform(Y_metab)
    n_samples = X.shape[0]
    if X.shape[1] > n_samples // 2:
        pca_x = _PCA(n_components=min(n_samples // 2, 50))
        X = pca_x.fit_transform(X)
    B, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    Y_hat = X @ B
    ss_total = np.sum(Y ** 2)
    ss_fitted = np.sum(Y_hat ** 2)
    r2_total = ss_fitted / ss_total
    pca_rda = _PCA(n_components=n_components)
    rda_coords = pca_rda.fit_transform(Y_hat)
    rda_var = pca_rda.explained_variance_ratio_ * r2_total
    return rda_coords, rda_var, r2_total


if EXPLORATORY:
    print('=' * 80)
    print(f'RDA: Redundancy Analysis (Species -> Metabolites) -- all {len(DATASETS)} datasets')
    print('=' * 80)

    fig, axes = plt.subplots(1, len(DATASETS), figsize=(8 * len(DATASETS), 6), squeeze=False)
    axes = axes.flatten()
    rda_results = []

    for i, ds in enumerate(DATASETS):
        spc = transformed_species[ds].set_index('Sample')
        mtb = transformed_mtb[ds].set_index('Sample')
        common = spc.index.intersection(mtb.index)
        X_spc = np.nan_to_num(spc.loc[common].values, nan=0.0)
        Y_mtb = np.nan_to_num(mtb.loc[common].values, nan=0.0)

        rda_coords, rda_var, r2 = rda_analysis(X_spc, Y_mtb)
        meta_sub = harmonized_meta[ds].set_index('Sample').loc[common]
        groups = meta_sub['Study.Group'].values
        unique_groups = sorted(set(groups))
        palette = sns.color_palette('Set2', len(unique_groups))
        color_map = {g: palette[j] for j, g in enumerate(unique_groups)}

        for group in unique_groups:
            mask = groups == group
            axes[i].scatter(rda_coords[mask, 0], rda_coords[mask, 1],
                             c=[color_map[group]], label=group, alpha=0.6, s=40)
            if mask.sum() >= 3:
                _draw_confidence_ellipse(axes[i], rda_coords[mask, 0],
                                          rda_coords[mask, 1], color_map[group])

        axes[i].set_xlabel(f'RDA1 ({rda_var[0]:.1%})')
        axes[i].set_ylabel(f'RDA2 ({rda_var[1]:.1%})')
        title_suffix = ' [EXPLORATORY]' if ds in DATASETS_IBD else ''
        axes[i].set_title(f'{ds}{title_suffix}\nR2={r2:.3f}', fontsize=9)
        axes[i].legend(fontsize=6)
        rda_results.append({'Dataset': ds, 'R2': round(r2, 4)})
        print(f'  {ds}: R2={r2:.4f}')

    for j in range(len(DATASETS), len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'RDA: Metabolite variance explained by species (all {len(DATASETS)} datasets)',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures' / 'qc' / 'rda_species_metabolites_v2.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    display(pd.DataFrame(rda_results))

---
## Step 7: Inter-Cohort Comparison & Feature-Level Analysis

> **Exploratory -- per fine-grained study group.** Yachida: Healthy vs each of Stage_0,
> Stage_I_II, Stage_III_IV (MP is excluded from the study; the Healthy-vs-PostSurgery
> sensitivity comparison is run separately in NB01b, so it never joins this FDR family).
> Sinha and Kim: Healthy vs CRC. These tables feed NB01b.
> Species-metabolite and cross-cohort correlations are computed in Notebook 02.

In [ ]:
if EXPLORATORY:
    da_results_mtb = {}
    da_results_spc = {}

    print('=' * 80)
    print(f'DIFFERENTIAL ABUNDANCE ANALYSIS -- {len(DATASETS)} cohorts (exploratory)')
    print('=' * 80)

    for ds in DATASETS:
        comparisons = COMPARISONS.get(ds, [])
        if not comparisons:
            continue
        da_results_mtb[ds] = {}
        da_results_spc[ds] = {}

        role_flag = '' if ds == DATASET_DISCOVERY else ' [validation cohort]'
        n_comps = len(comparisons)
        pa_features = protected_by_ds[ds]

        # One figure per dataset: rows = comparisons, cols = Metabolites | Species
        fig, axes = plt.subplots(
            n_comps, 2,
            figsize=(14, 4.5 * n_comps),
            squeeze=False,
        )

        for row_idx, (group1, group2) in enumerate(comparisons):
            label = f'{group1}_vs_{group2}'
            da_mtb = differential_abundance(
                transformed_mtb[ds], harmonized_meta[ds], group1, group2)
            da_spc = differential_abundance(
                transformed_species[ds], harmonized_meta[ds], group1, group2)
            da_results_mtb[ds][label] = da_mtb
            da_results_spc[ds][label] = da_spc

            n_sig_mtb = int(da_mtb['Significant'].sum()) if not da_mtb.empty else 0
            n_sig_spc = int(da_spc['Significant'].sum()) if not da_spc.empty else 0
            print(f'  {ds} {label}{role_flag}: '
                  f'{n_sig_mtb} sig metabolites, {n_sig_spc} sig species')

            ax_mtb, ax_spc = axes[row_idx, 0], axes[row_idx, 1]

            # fc_col='MeanDiff': differential_abundance() reports a linear-space mean
            # difference, not a Log2FC -- the old Log2FC = log2(ratio-of-already-
            # transformed-means) was mathematically invalid and is gone from utils.
            if not da_mtb.empty:
                volcano_plot(da_mtb,
                             title=f'Metabolites: {group1} vs {group2}',
                             fc_col='MeanDiff',
                             highlight_features=pa_features,
                             ax=ax_mtb)
            else:
                ax_mtb.set_title(f'Metabolites: {group1} vs {group2} (no data)', fontsize=9)
                ax_mtb.axis('off')

            if not da_spc.empty:
                volcano_plot(da_spc,
                             title=f'Species: {group1} vs {group2}',
                             fc_col='MeanDiff',
                             ax=ax_spc)
            else:
                ax_spc.set_title(f'Species: {group1} vs {group2} (no data)', fontsize=9)
                ax_spc.axis('off')

        fig.suptitle(f'{ds}{role_flag} Differential Abundance -- Metabolites (left) | Species (right)',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path = RESULTS_DIR / 'figures' / 'differential' / f'{ds}_da_subplots_v2.png'
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f'  Saved: {save_path.name}')

    # Guard: no primary DA comparison may involve the sensitivity group.
    for ds, comps in da_results_mtb.items():
        for label in comps:
            assert 'PostSurgery' not in label and 'HS' not in label.split('_vs_'), \
                f'sensitivity group leaked into the primary DA family: {ds} {label}'
    print('\nGuard OK: no PostSurgery comparison in the primary DA family.')

**Species-metabolite correlations and cross-cohort consistency** are no longer computed
here. They made the old pickle ~800 MB, nothing downstream read them, and NB02 Step 7 does
the confirmatory version (Yachida + Sinha, genus-level cross-cohort matching).

---
## Save Preprocessed Data for Notebook 02

In [ ]:
# Guard against EXPLORATORY=False: these keys are only populated when the exploratory
# Step 6/7 cells above actually ran. locals().get(...) returns None instead of raising
# NameError if EXPLORATORY was False (or if those cells were never executed), so this
# cell -- and therefore the whole notebook -- behaves correctly in BOTH cases.
preprocessed = {
    'transformed_species': transformed_species,
    'transformed_mtb':     transformed_mtb,
    'filtered_species':    filtered_species,
    'filtered_mtb':        filtered_mtb,
    'harmonized_meta':     harmonized_meta,
    'polyamine_columns':   polyamine_columns,
    'polyamine_proxies':   polyamine_proxies,
    'mtb_kegg_map':        mtb_kegg_map,
    'da_results_spc':      locals().get('da_results_spc'),
    'da_results_mtb':      locals().get('da_results_mtb'),
    'correlation_results': None,   # moved to NB02 (see Step 7 note)
    'consistent_all':      None,
    'mtb_prescaled':       MTB_PRESCALED,
    'excluded_study_groups':   utils.EXCLUDED_STUDY_GROUPS,
    'sensitivity_groups':      utils.SENSITIVITY_GROUPS,
    'stage_group_order':       STAGE_GROUP_ORDER,
    'sensitivity_stage_groups': utils.SENSITIVITY_STAGE_GROUPS,
    'datasets':                DATASETS,
    'dataset_discovery':       DATASET_DISCOVERY,
    'datasets_validation':     DATASETS_VALIDATION,
}

save_path = RESULTS_DIR / 'intermediate' / 'preprocessed_data_v2.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(preprocessed, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Preprocessed data saved to: {save_path}')
print(f'File size: {save_path.stat().st_size / 1e6:.1f} MB')
assert save_path.stat().st_size < 200e6, 'preprocessed pickle unexpectedly large'
print(f'EXPLORATORY = {EXPLORATORY}')

print('\n' + '=' * 80)
print('PREPROCESSING COMPLETE')
print('=' * 80)
print(f'\nCohorts preprocessed: {len(DATASETS)}')
print(f'  Discovery : {DATASET_DISCOVERY}')
print(f'  Validation: {DATASETS_VALIDATION}')

for ds in DATASETS:
    n_spc = len(transformed_species[ds].columns) - 1
    n_mtb = len(transformed_mtb[ds].columns) - 1
    n_pa = len(polyamine_columns.get(ds, {}))
    n_px = len(polyamine_proxies.get(ds, {}))
    role = 'discovery ' if ds == DATASET_DISCOVERY else 'validation'
    print(f'  {ds:22s} [{role}]: {len(transformed_species[ds]):3d} samples, '
          f'{n_spc:5d} species, {n_mtb:4d} metabolites, {n_pa} polyamines (+{n_px} proxy)')

# Final group accounting for the discovery cohort, so the run log records the
# numbers every downstream notebook depends on.
_ym = harmonized_meta[DATASET_DISCOVERY]
print(f'\n{DATASET_DISCOVERY} group accounting:')
print(f'  loaded      n={len(_ym)}')
print(f'  primary     n={int(primary_mask(_ym).sum())}  ({STAGE_GROUP_ORDER})')
print(f'  sensitivity n={int(sensitivity_mask(_ym).sum())}  ({utils.SENSITIVITY_STAGE_GROUPS})')
print(f'  MP excluded at load: {utils.EXCLUDED_STUDY_GROUPS[DATASET_DISCOVERY]}')